# View

In [2]:
import open3d as o3d

# Path to your PLY file
ply_path = r"C:\Users\cmrit\OneDrive - radometech.com\Documents\3D_Final\3d-output\03_sliced_between_Rs.ply"

# Load point cloud
pcd = o3d.io.read_point_cloud(ply_path)

# Print basic info
print(pcd)
print("Number of points:", len(pcd.points))

# Visualize
o3d.visualization.draw_geometries([pcd])

PointCloud with 1809744 points.
Number of points: 1809744


# Final code

In [ ]:
import cv2
import numpy as np
import open3d as o3d
from plyfile import PlyData
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction


# -----------------------------
# LOAD MODEL (GENERIC)
# -----------------------------
def load_model(model_path, device="cpu", conf=0.3):
    return AutoDetectionModel.from_pretrained(
        model_type="yolov8",
        model_path=model_path,
        confidence_threshold=conf,
        device=device
    )


# -----------------------------
# STEP 1: PLY → 2D → DETECT ROI → SLICE
# -----------------------------
def slice_point_cloud(ply_path, model):

    pcd = o3d.io.read_point_cloud(ply_path)
    pts = np.asarray(pcd.points)

    x, y, z = pts[:, 0], pts[:, 1], pts[:, 2]

    # project to 2D
    img = np.zeros((1024, 1024), dtype=np.uint8)
    xi = ((x - x.min()) / (x.max() - x.min()) * 1023).astype(int)
    yi = ((y - y.min()) / (y.max() - y.min()) * 1023).astype(int)

    img[1023 - yi, xi] = 255

    # detect ROI (R regions)
    result = get_sliced_prediction(img, model, 512, 512, 0.2, 0.2)

    boxes = [ann["bbox"] for ann in result.to_coco_annotations()]
    boxes = sorted(boxes, key=lambda b: b[1])[:2]

    # convert to world Y
    y1 = boxes[0][1]
    y2 = boxes[1][1]

    mask = (y >= y1) & (y <= y2)
    return pts[mask]


# -----------------------------
# STEP 2: PROJECT TO IMAGE
# -----------------------------
def project_to_2d(points, W=2000, H=10000):

    x, y, z = points[:, 0], points[:, 1], points[:, 2]

    xi = ((x - x.min()) / (x.max() - x.min()) * (W - 1)).astype(int)
    yi = ((y.max() - y) / (y.max() - y.min()) * (H - 1)).astype(int)

    img = np.zeros((H, W), dtype=np.float32)
    img[yi, xi] = z

    img = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    return img


# -----------------------------
# STEP 3: DETECT MASK
# -----------------------------
def detect_mask(image, model):

    result = get_sliced_prediction(image, model, 512, 512, 0.2, 0.2)

    mask = np.zeros(image.shape[:2], dtype=np.uint8)

    for obj in result.object_prediction_list:
        if obj.mask:
            for poly in obj.mask.segmentation:
                pts = np.array(poly, dtype=np.int32).reshape(-1, 2)
                cv2.fillPoly(mask, [pts], 255)

    return mask


# -----------------------------
# STEP 4: ALIGN (CENTROID)
# -----------------------------
def align_images(img1, mask1, mask2):

    def centroid(m):
        M = cv2.moments(m)
        return (M["m10"]/M["m00"], M["m01"]/M["m00"])

    c1 = centroid(mask1)
    c2 = centroid(mask2)

    dx = c2[0] - c1[0]
    dy = c2[1] - c1[1]

    M = np.float32([[1, 0, dx], [0, 1, dy]])
    aligned = cv2.warpAffine(img1, M, (img1.shape[1], img1.shape[0]))

    return aligned


# -----------------------------
# STEP 5: DEFECT → 3D → DIMENSIONS
# -----------------------------
def compute_dimensions(points, mask, W, H):

    x, y = points[:, 0], points[:, 1]

    xi = ((x - x.min()) / (x.max() - x.min()) * (W - 1)).astype(int)
    yi = ((y.max() - y) / (y.max() - y.min()) * (H - 1)).astype(int)

    inside = mask[yi, xi] > 0
    defect_pts = points[inside]

    if len(defect_pts) < 3:
        return None

    centered = defect_pts - np.mean(defect_pts, axis=0)
    _, _, vt = np.linalg.svd(centered)

    proj = centered @ vt.T
    dims = proj.max(axis=0) - proj.min(axis=0)

    return sorted(dims, reverse=True)[:3]


# -----------------------------
# MAIN PIPELINE (ABSTRACT)
# -----------------------------
def run_pipeline(ply_path, image, model_r, model_defect):

    # 1. Slice point cloud using ROI
    sliced = slice_point_cloud(ply_path, model_r)

    # 2. Project to 2D
    rendered = project_to_2d(sliced)

    # 3. Detect alignment mask
    mask_rendered = detect_mask(rendered, model_r)
    mask_original = detect_mask(image, model_r)

    # 4. Align
    aligned = align_images(image, mask_original, mask_rendered)

    # 5. Detect defect
    defect_mask = detect_mask(aligned, model_defect)

    # 6. Compute dimensions
    dims = compute_dimensions(sliced, defect_mask, 2000, 10000)

    return dims